# VQA Enrichment — Qwen2.5-VL-7B (GPU)
Запускать ячейки ПО ПОРЯДКУ

In [ ]:
%pip install "transformers==4.45.2" "accelerate==0.34.2" Pillow tqdm

## ⚠️ Kernel → Restart Kernel (обязательно после pip install)

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
print('Transformers OK')

model = Qwen2VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2.5-VL-7B-Instruct',
    torch_dtype=torch.float16,
    device_map='auto'
)
processor = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct')
print('Model loaded!')

In [ ]:
import json, time, os
from PIL import Image
from tqdm.notebook import tqdm

PROMPT = '''Context: Caption: "{caption}", OCR: "{ocr_text}", Tone: "{tone}"

Return JSON with:
1. "ocr_normalized" - cleaned OCR text
2. "objects_detailed" - list 5-15 visible objects
3. "relations" - up to 5 triples ["subj", "rel", "obj"]
4. "required_context" - {{"is_required": bool, "what": "..."}}
5. "vqa" - list of {{"q":...,"a":...}} for: What text? What depicted? What happening? What is joke? Who is audience? One search sentence.

JSON only, no markdown.'''

IMG = {}
for root, dirs, files in os.walk('images_extracted'):
    for f in files:
        if not f.startswith('._'):
            IMG[f] = os.path.join(root, f)
print(f'{len(IMG)} images cached')

def query(img_path, rec):
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    if max(w, h) > 512:
        r = 512 / max(w, h)
        img = img.resize((int(w * r), int(h * r)), Image.LANCZOS)
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': img},
        {'type': 'text', 'text': PROMPT.format(
            caption=rec.get('caption', ''),
            ocr_text=rec.get('ocr_text', '')[:200],
            tone=rec.get('tone', ''))}
    ]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = processor(text=[text], images=[img], padding=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=800, temperature=0.3, do_sample=True)
    return processor.batch_decode(out[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0]

def parse(raw):
    if not raw: return None
    t = raw.strip()
    if '```json' in t: t = t.split('```json')[1]
    if '```' in t: t = t.split('```')[0]
    s, e = t.find('{'), t.rfind('}') + 1
    if s != -1 and e > s:
        try: return json.loads(t[s:e])
        except: pass
    return None

print('Functions ready!')

In [ ]:
# Тест на 3 картинках
records = [json.loads(l) for l in open('vqa_annotations.jsonl')]
print(f'Records: {len(records)}')

for rec in records[:3]:
    p = IMG.get(rec['filename'])
    if not p: print(f'SKIP {rec["filename"]}'); continue
    t0 = time.time()
    raw = query(p, rec)
    dt = time.time() - t0
    res = parse(raw)
    print(f'\n--- {rec["filename"]} ({dt:.1f}s) ---')
    print(json.dumps(res, indent=2, ensure_ascii=False)[:400] if res else f'FAIL: {raw[:200]}')

In [ ]:
# Полный прогон
OUT = 'vqa_annotations_v2.jsonl'
done = set()
if os.path.exists(OUT):
    done = {json.loads(l).get('filename', '') for l in open(OUT)}
remaining = [r for r in records if r['filename'] not in done]
print(f'Done: {len(done)}, remaining: {len(remaining)}')

ok = fail = skip = 0
t0 = time.time()
with open(OUT, 'a') as f:
    for rec in tqdm(remaining):
        p = IMG.get(rec['filename'])
        if not p:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            skip += 1; continue
        try:
            raw = query(p, rec)
            res = parse(raw)
        except Exception as e:
            print(f'ERROR {rec["filename"]}: {e}')
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            fail += 1; continue
        if res:
            m = {**rec}
            m['ocr_normalized'] = res.get('ocr_normalized', '')
            m['objects_detailed'] = res.get('objects_detailed', rec.get('objects', []))
            m['relations'] = res.get('relations', [])
            m['required_context'] = res.get('required_context', {'is_required': False, 'what': ''})
            m['vqa'] = res.get('vqa', [])
            f.write(json.dumps(m, ensure_ascii=False) + '\n')
            ok += 1
        else:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            fail += 1
        f.flush()
        n = ok + fail + skip
        if n % 100 == 0 and n > 0:
            s = (time.time() - t0) / n
            eta = s * (len(remaining) - n) / 3600
            print(f'[{n}/{len(remaining)}] {s:.1f}s/img ETA:{eta:.1f}h ok:{ok} fail:{fail}')
print(f'\nDone {(time.time() - t0) / 3600:.1f}h ok:{ok} fail:{fail} skip:{skip}')

In [ ]:
# Проверка результатов
data = [json.loads(l) for l in open(OUT)]
vqa_count = sum(1 for d in data if d.get('vqa'))
print(f'Total: {len(data)}, with VQA: {vqa_count} ({vqa_count * 100 // len(data)}%)')